Responsibilities:

- load bronze tables
- 
- apply DQ framework
- 
- deduplication
- 
- referential validation.

Important:

👉 One notebook handles ALL tables.

NOT multiple notebooks.

In [0]:
from pyspark.sql.functions import col

In [0]:
silver_config = {

   "orders": {
       "primary_key": ["order_id"],
       "not_null": ["order_id","customer_id"]
   },

   "order_items": {
       "primary_key": ["order_id","order_item_id"],
       "not_null": ["order_id","product_id"]
   },

   "order_payments": {
       "primary_key": ["order_id","payment_sequential"],
       "not_null": ["order_id"]
   },

   "order_reviews": {
       "primary_key": ["review_id"],
       "not_null": ["review_id"]
   },

   "customers": {
       "primary_key": ["customer_id"],
       "not_null": ["customer_id"]
   },

   "products": {
       "primary_key": ["product_id"],
       "not_null": ["product_id"]
   },

   "sellers": {
       "primary_key": ["seller_id"],
       "not_null": ["seller_id"]
   }

}


In [0]:
def load_bronze(table_name):

    return spark.table(f"e_comm_databricks.pipeline_bronze.{table_name}")


In [0]:
def deduplicate(df, keys):

    return df.dropDuplicates(keys)


In [0]:
def remove_nulls(df, columns):

    from pyspark.sql.functions import col

    for c in columns:
        df = df.filter(col(c).isNotNull())

    return df


In [0]:
def write_silver(df, table_name):

    df.write \
      .format("delta") \
      .mode("overwrite") \
      .saveAsTable(f"e_comm_databricks.pipeline_silver.{table_name}")


In [0]:
def process_table(table_name, rules):

    print(f"Processing {table_name}")

    df = load_bronze(table_name)

    df = deduplicate(df, rules["primary_key"])

    df = remove_nulls(df, rules["not_null"])

    write_silver(df, table_name)


In [0]:
for table, rules in silver_config.items():

    process_table(table, rules)
